一、知识产权年鉴 专利执法、查处执法、地区专利情况数据清洗

In [ ]:
import pandas as pd
import numpy as np
import os

base_path = r"D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据"

# 1. 读取所有年份的专利执法数据（h1.xls） 
def load_patent_enforcement():
    """读取各地区管理专利工作的部门专利执法统计表（h1.xls）"""
    all_data = []
    
    for year in range(2008, 2019):  # 2008-2018
        file_path = os.path.join(base_path, str(year), "h", "h1.xls")
        if os.path.exists(file_path):
            # 读取数据
            df = pd.read_excel(file_path, usecols=[1, 2, 3, 4], header=2)
            df = df.drop([0, 1])
            for i in range(0, len(df)):
                if i % 2 == 0:
                    city = df.iloc[i,0]
                if i % 2 != 0:
                    df.iloc[i,0] = city
            df = df.iloc[1::2].reset_index(drop=True)
            df.columns = ['地区', '年份', '立案', '结案']
            all_data.append(df)
            print(f"已读取：{file_path}")
        else:
            print(f"警告：文件不存在 {file_path}")
    
    result = pd.concat(all_data, ignore_index=True)
    return result

# 执行读取
df_enforcement = load_patent_enforcement()
print(f"\n专利执法数据读取完成，共 {len(df_enforcement)} 条记录")
print(df_enforcement.head())

# 计算结案/立案比值
df_enforcement['结案率'] = df_enforcement['结案'] / df_enforcement['立案']
# 处理异常值
df_enforcement['结案率'] = df_enforcement['结案率'].replace([np.inf, -np.inf], 0).fillna(0)
df_enforcement.loc[df_enforcement['结案率'] > 1, '结案率'] = 1

# 2. 读取查处假冒专利数据（h5.xls）
def load_counterfeit():
    """读取各地区查处假冒专利执法统计表（h5.xls）"""
    all_data = []
    
    for year in range(2008, 2019):
        file_path = os.path.join(base_path, str(year), "h", "h5.xls")
        if os.path.exists(file_path):
            df = pd.read_excel(file_path, usecols=[1, 3], header=1)
            if year < 2010:#2008-2009
                df = df.drop(0)
            else:#2010-2018
                df = df.drop([0,1])
            df.columns = ['地区', '假冒数量']
            df['年份'] = year
            all_data.append(df)
            print(f"已读取：{file_path}")
        else:
            print(f"警告：文件不存在 {file_path}")
    
    result = pd.concat(all_data, ignore_index=True)
    return result

df_counterfeit = load_counterfeit()
print(f"\n查处假冒数据读取完成，共 {len(df_counterfeit)} 条记录")
print(df_counterfeit.head())

# 3. 读取地区专利情况（a4、b3、c2）
def load_patent_status():
    """读取三种专利的申请量、授权量、有效量"""
    all_apply = []    # 申请量
    all_grant = []    # 授权量
    all_valid = []    # 有效量
    all_pro = []      # 职务专利百分比
    for year in range(2008, 2019):
        # 读取申请量 a4.xls
        apply_path = os.path.join(base_path, str(year), "a", "a4.xls")
        if os.path.exists(apply_path):
            if year < 2010:#2008-2009
                df_apply = pd.read_excel(apply_path, usecols=[1, 5, 6, 7, 8], header=2)
            elif year > 2016:#2017-2018
                df_apply = pd.read_excel(apply_path, usecols=[1, 2, 3, 4, 5], header=2)
            else:#2010-2016
                df_apply = pd.read_excel(apply_path, usecols=[1, 8, 9, 10, 11], header=2)
            df_apply.iloc[:,1] = df_apply.iloc[:,2] + df_apply.iloc[:,3] + df_apply.iloc[:,4]
            df_apply = df_apply.drop(0)
            df_apply.columns = ['地区', '专利_申请量', '发明_申请量', '实用新型_申请量', '外观设计_申请量']
            df_apply['年份'] = year
            all_apply.append(df_apply)
            print(f"已读取：{apply_path}")

    for year in range(2008, 2019):
        # 读取授权量 b3.xls
        grant_path = os.path.join(base_path, str(year), "b", "b3.xls")
        if os.path.exists(grant_path):
            df_grant = pd.read_excel(grant_path, usecols=[1, 5, 6, 7, 8], header=2)
            df_grant.iloc[:,1] = df_grant.iloc[:,2] + df_grant.iloc[:,3] + df_grant.iloc[:,4]
            df_grant = df_grant.drop(0)
            df_grant.columns = ['地区', '专利_授权量', '发明_授权量', '实用新型_授权量', '外观设计_授权量']
            df_grant['年份'] = year
            all_grant.append(df_grant)
            print(f"已读取：{grant_path}")

    for year in range(2008, 2019):
        # 读取有效量 c2.xls
        valid_path = os.path.join(base_path, str(year), "c", "c2.xls")
        if os.path.exists(valid_path):
            df_valid = pd.read_excel(valid_path, usecols=[1, 4, 5, 6], header=2)
            df_valid = df_valid.drop(0)
            if year == 2008:
                df_valid = df_valid.drop([35, 36, 37])
            df_valid.columns = ['地区', '发明_有效量', '实用新型_有效量', '外观设计_有效量']
            df_valid['年份'] = year
            all_valid.append(df_valid)
            print(f"已读取：{valid_path}")
    
    
    for year in range(2008, 2019):
        # 读取职务专利百分比 c3.xls
        pro_path = os.path.join(base_path, str(year), "c", "c3.xls")
        if os.path.exists(valid_path):
            df_pro = pd.read_excel(pro_path, usecols=[1, 3], header=2)
            df_pro = df_pro.drop([0, 1])
            for i in range(0, len(df_pro)):
                if i % 2 == 0:
                    city = df_pro.iloc[i,0]
                if i % 2 != 0:
                    df_pro.iloc[i,0] = city
            df_pro = df_pro.iloc[1::2].reset_index(drop=True)
            df_pro.columns = ['地区', '职务专利百分比']
            df_pro['年份'] = year
            all_pro.append(df_pro)
            print(f"已读取：{pro_path}")


    # 合并四个数据集
    df_apply_all = pd.concat(all_apply, ignore_index=True)
    df_grant_all = pd.concat(all_grant, ignore_index=True)
    df_valid_all = pd.concat(all_valid, ignore_index=True)
    df_pro_all = pd.concat(all_pro, ignore_index=True)

    # 合并成一个表
    df_status = pd.merge(df_apply_all, df_grant_all, on=['地区', '年份'])
    df_status = pd.merge(df_status, df_valid_all, on=['地区', '年份'])
    df_status = pd.merge(df_status, df_pro_all, on=['地区', '年份'])
    
    # 计算比值（处理除零）
    df_status['发明专利授权率'] = df_status['发明_授权量'] / df_status['发明_申请量'].replace(0, np.nan)
    df_status['实用新型专利授权率'] = df_status['实用新型_授权量'] / df_status['实用新型_申请量'].replace(0, np.nan)
    df_status['外观设计专利授权率'] = df_status['外观设计_授权量'] / df_status['外观设计_申请量'].replace(0, np.nan)
    df_status['专利授权率'] = df_status['专利_授权量'] / df_status['专利_申请量'].replace(0, np.nan)

    df_status['发明专利申请比例'] = df_status['发明_申请量'] / df_status['专利_申请量'].replace(0, np.nan)
    df_status['实用新型专利申请比例'] = df_status['实用新型_申请量'] / df_status['专利_申请量'].replace(0, np.nan)
    df_status['外观设计专利申请比例'] = df_status['外观设计_申请量'] / df_status['专利_申请量'].replace(0, np.nan)
    
    df_status['发明专利授权比例'] = df_status['发明_授权量'] / df_status['专利_授权量'].replace(0, np.nan)
    df_status['实用新型专利授权比例'] = df_status['实用新型_授权量'] / df_status['专利_授权量'].replace(0, np.nan)
    df_status['外观设计专利授权比例'] = df_status['外观设计_授权量'] / df_status['专利_授权量'].replace(0, np.nan)
    
    df_status['发明专利有效比例'] = df_status['发明_有效量'] / (df_status['发明_有效量'] + df_status['实用新型_有效量'] + df_status['外观设计_有效量']).replace(0, np.nan)
    df_status['实用新型专利有效比例'] = df_status['实用新型_有效量'] / (df_status['发明_有效量'] + df_status['实用新型_有效量'] + df_status['外观设计_有效量']).replace(0, np.nan)
    df_status['外观设计专利有效比例'] = df_status['外观设计_有效量'] / (df_status['发明_有效量'] + df_status['实用新型_有效量'] + df_status['外观设计_有效量']).replace(0, np.nan)
    
    return df_status

df_patent_status = load_patent_status()
print(f"\n地区专利情况数据读取完成，共 {len(df_patent_status)} 条记录")

# 4. 整合所有数据 
# 确保年份列为整数类型
df_enforcement['年份'] = df_enforcement['年份'].astype(int)
df_counterfeit['年份'] = df_counterfeit['年份'].astype(int)
df_patent_status['年份'] = df_patent_status['年份'].astype(int)
# 合并执法数据、假冒数据、专利情况数据
df_final = pd.merge(df_enforcement, df_counterfeit, on=['地区', '年份'], how='outer')
df_final = pd.merge(df_final, df_patent_status, on=['地区', '年份'], how='outer')
df_final = df_final.fillna(0)# 空值补零
# 设置年份和地区为双重行索引
df_final.set_index(['年份', '地区'], inplace=True)

# 按年份和地区排序
df_final.sort_index(inplace=True)

# 保存整合后的数据
output_path = r"ex2_1知识产权年鉴数据.xlsx"
df_final.to_excel(output_path)
print(f"\n数据整合完成，已保存至：{output_path}")
print(f"数据形状：{df_final.shape}")
print("\n数据前5行预览：")
print(df_final.head())


已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2008\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2009\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2010\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2011\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2012\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2013\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2014\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2015\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2016\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2017\h\h1.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2018\h\h1.xls

专利执法数据读取完成，共 343 条记录
    地区    年份  立案  结案
0   北京  2008  42  27
1   天津  2008  10   8
2   河北  2008  19  16
3   山西  2008   3   2
4  内蒙古  2008  10   9
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2008\h\h5.xls
已读取：D:\Limmo\Python数分\实验数据\专利统计数据\2008-2018知识产权年鉴数据\2009\h\h

C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_12708\402422996.py:182: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df_final.fillna(0)# 空值补零



数据整合完成，已保存至：ex2_1知识产权年鉴数据.xlsx
数据形状：(546, 29)

数据前5行预览：
            立案    结案       结案率 假冒数量  专利_申请量  发明_申请量  实用新型_申请量  外观设计_申请量  \
年份   地区                                                                   
2008 上海   14.0  27.0  1.000000    0   52835   17831     14325     20679   
     云南    8.0   4.0  0.500000    0    4089    1474      1389      1226   
     内蒙古  10.0   9.0  0.900000    0    2221     695       980       546   
     北京   42.0  27.0  0.642857    0   43508   28394     11157      3957   
     厦门    0.0   0.0  0.000000    0    3336     848      1509       979   

          专利_授权量  发明_授权量  ...     专利授权率  发明专利申请比例  实用新型专利申请比例  外观设计专利申请比例  \
年份   地区                   ...                                               
2008 上海    24468    4258  ...  0.463102  0.337485    0.271127    0.391388   
     云南     2021     383  ...  0.494253  0.360479    0.339692    0.299829   
     内蒙古    1328     140  ...  0.597929  0.312922    0.441243    0.245835   
     北京    17747    6478  ...  0

二、人民日报数据清洗

In [ ]:
import pandas as pd
import re

# 1. 读取数据
df = pd.read_excel("D:\Limmo\Python数分\实验数据\舆情数据\人民日报.xlsx")

# 2. 新增年份列，去除标题列
df['年份'] = df['标题'].astype(str).str[:4]
df = df.drop(columns=['标题'])

# 3. 定义清洗函数和关键词匹配
def clean_text(text):
    """清洗文本：去除多余空白和标点符号"""
    text = re.sub(r'\s+', '', text)          # 去除空白字符
    text = re.sub(r'[，。？；：''、！（）《》【】“”]', '', text)  # 去除标点
    return text

keywords = ['知识产权', '专利', '商标', '版权', '著作权']

def is_ip_article(text):
    """判断是否为知识产权相关文章"""
    return any(kw in text for kw in keywords)

# 4. 按年统计
results = []

for year in df['年份'].unique():
    year_data = df[df['年份'] == year].copy()
    
    # 清洗内容并判断是否为知识产权文章
    year_data['清洗后内容'] = year_data['正文'].astype(str).apply(clean_text)
    year_data['是否为知识产权文章'] = year_data['清洗后内容'].apply(is_ip_article)
    
    ip_data = year_data[year_data['是否为知识产权文章']]
    
    ip_count = len(ip_data)
    ip_total_words = ip_data['清洗后内容'].str.len().sum()
    total_articles = len(year_data)
    total_words = year_data['清洗后内容'].str.len().sum()
    
    results.append({
        '年份': year,
        '总发文量': total_articles,
        '知识产权文章数量': ip_count,
        '全部文章总字数': total_words,
        '知识产权文章总字数': ip_total_words
    })

result_df = pd.DataFrame(results)

# 5. 计算指标
result_df['知识产权文章数量占比'] = result_df['知识产权文章数量'] / result_df['总发文量']
result_df['知识产权文章字数占比'] = result_df['知识产权文章总字数'] / result_df['全部文章总字数']
result_df['知识产权文章平均字数'] = result_df['知识产权文章总字数'] / result_df['知识产权文章数量']

# 6. 输出结果
print(result_df)

# 7. 保存结果
result_df.to_excel("ex2_2人民日报_知识产权统计分析.xlsx", index=False)
print("\n结果已保存至：ex2_2人民日报_知识产权统计分析.xlsx")

<>:5: SyntaxWarning: invalid escape sequence '\L'
<>:5: SyntaxWarning: invalid escape sequence '\L'
C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_4116\4184734435.py:5: SyntaxWarning: invalid escape sequence '\L'
  df = pd.read_excel("D:\Limmo\Python数分\实验数据\舆情数据\人民日报.xlsx")


     年份   总发文量  知识产权文章数量   全部文章总字数  知识产权文章总字数  知识产权文章数量占比  知识产权文章字数占比  \
0  2018  33233      1463  37135863    3578931    0.044023    0.096374   
1  2019  19943       957  24604307    2728001    0.047987    0.110875   

    知识产权文章平均字数  
0  2446.295967  
1  2850.575758  

结果已保存至：ex2_2人民日报_知识产权统计分析.xlsx


三、微博数据清洗

In [ ]:
import pandas as pd

# 1. 读取微博数据
data = pd.read_excel('D:\Limmo\Python数分\实验数据\舆情数据\北京知识产权局_微博.xlsx')
print("原始数据前5行：")
print(data.head())

# 2. 选取需要的列
data1 = data[['日期', '转发', '评论', '点赞', 'text']].copy()

# 3. 清洗微博内容列：去掉不需要的字符串
# 定义需要去掉的模式
patterns = [
    r'\.\.\.',           # 省略号
    r'\\u200b',          # 零宽空格
    r'O网页链接',        # 网页链接标识
    r'\\u3000'           # 全角空格
]

for pattern in patterns:
    data1['text'] = data1['text'].str.replace(pattern, '', regex=True)

# 去掉所有HTML标签和多余空白
data1['text'] = data1['text'].str.replace(r'<[^>]+>', '', regex=True)
data1['text'] = data1['text'].str.strip()

# 4. 处理日期列：只保留年份
data1['日期'] = pd.to_datetime(data1['日期'], errors='coerce')
data1['日期'] = data1['日期'].dt.year
data1.rename(columns={'日期':'年份'},inplace=True)
# 5. 计算每年的发文量，并新增发文量列
yearly_counts = data1['年份'].value_counts()
data1['发文量'] = data1['年份'].map(yearly_counts)

# 6. 计算知识产权平均字数（衡量重视程度）
keywords = ['知识产权', '专利', '商标', '版权', '著作权']

def is_ip_related(text):
    if pd.isna(text):
        return False
    return any(kw in str(text) for kw in keywords)

data1['是否知识产权'] = data1['text'].apply(is_ip_related)
data1['微博字数'] = data1['text'].apply(lambda x: len(str(x)) if pd.notna(x) else 0)

# 按年份统计
yearly_stats = data1.groupby('年份').agg(
    知识产权类字数=('微博字数', lambda x: x[data1.loc[x.index, '是否知识产权']].sum()),
    全部字数=('微博字数', 'sum'),
    知识产权类发文量=('是否知识产权', 'sum'),
    全部发文量=('text', 'count')
)
yearly_stats['知识产权平均字数'] = yearly_stats['知识产权类字数'] / yearly_stats['知识产权类发文量']
yearly_stats['知识产权平均字数'] = yearly_stats['知识产权平均字数'].fillna(0)

# 新增知识产权平均字数列
data1['知识产权平均字数'] = data1['年份'].map(yearly_stats['知识产权平均字数'])
# 修改列名
data1.rename(columns={'转发':'转发数', '评论':'评论数', '点赞':'点赞数', 'text':'微博内容'},inplace=True)

# 7. 保存清洗后的数据
data1.to_excel('ex2_3北京知识产权局_微博_清洗后数据.xlsx', index=False)

print("清洗完成，结果前5行：")
print(data1.head())
print("结果已保存到 ex2_3北京知识产权局_微博_清洗后数据.xlsx")

<>:4: SyntaxWarning: invalid escape sequence '\L'
<>:4: SyntaxWarning: invalid escape sequence '\L'
C:\Users\admin.DESKTOP-K6PI601\AppData\Local\Temp\ipykernel_4116\2214472007.py:4: SyntaxWarning: invalid escape sequence '\L'
  data = pd.read_excel('D:\Limmo\Python数分\实验数据\舆情数据\北京知识产权局_微博.xlsx')


原始数据前5行：
   Unnamed: 0         日期  转发  评论  点赞  \
0           0 2026-05-15   0   0   0   
1           1 2026-05-15   0   1   1   
2           2 2026-05-14   0   0   0   
3           3 2026-05-13   0   0   2   
4           4 2026-05-12   0   0   2   

                                                text  
0  【北京市知识产权局开展“树立和践行正确政绩观”主题党日活动】为深入学习贯彻习近平总书记关于树...  
1                     【北京市知识产权局关于组织申报2026年专利导航项目的通知】  
2                    发布了头条文章：《第二十八届科博会知识产权服务业展区亮点纷呈》  
3  【聚焦科博会知识产权保护 赋能首都新质生产力发展】5月8日至10日，第二十八届中国北京国际科...  
4  【延庆区：品本土特色 识正规品牌】5月10日是第十个“中国品牌日”。为进一步提升辖区本土品牌...  
清洗完成，结果前5行：
     年份  转发数  评论数  点赞数                                               微博内容  \
0  2026    0    0    0  【北京市知识产权局开展“树立和践行正确政绩观”主题党日活动】为深入学习贯彻习近平总书记关于树...   
1  2026    0    1    1                     【北京市知识产权局关于组织申报2026年专利导航项目的通知】   
2  2026    0    0    0                    发布了头条文章：《第二十八届科博会知识产权服务业展区亮点纷呈》   
3  2026    0    0    2  【聚焦科博会知识产权保护 赋能首都新质生产力发展】5月8日至10日，第二十八届中国北京国际科...   
4  2026    0    0    2 

四、中国统计年鉴数据清洗

In [ ]:
import pandas as pd
import numpy as np

# （一）中国统计年鉴数据清洗与整合（GDP数据）

# 1. 读取2009-2013年GDP数据
df1 = pd.read_excel(r'D:\Limmo\Python数分\实验数据\宏观数据\GDP\2009-2013地区生产总值.xls', header=None)
# 根据数据结构进行清洗，数据从第2行开始，需要提取地区和年份数据
df1 = df1.iloc[1:, [0, 2, 3, 4, 5, 6]]  # 取地区和2009-2013年数据
df1.columns = ['地区', '2009', '2010', '2011', '2012', '2013']
# 重置索引并去除空值
df1 = df1.dropna().reset_index(drop=True)
print("2009-2013年GDP数据：")
print(df1.head())
print()

# 2. 读取2014-2017年GDP数据
df2_list = []
years = [2014, 2015, 2016, 2017]

for year in years:
    file_path = f'D:\Limmo\Python数分\实验数据\宏观数据\GDP\{year}各地区生产总值.xls'
    temp_df = pd.read_excel(file_path)
    # 清洗数据，提取地区和当年GDP
    temp_df = temp_df.iloc[5:, [0, 2]]  # 第0列是地区，第2列是GDP
    temp_df.columns = ['地区', str(year)]
    df2_list.append(temp_df)

# 合并2014-2017年数据
from functools import reduce
df2 = reduce(lambda left, right: pd.merge(left, right, on='地区', how='outer'), df2_list)
print("2014-2017年GDP数据：")
print(df2.head())
print()

# 3. 读取2018年GDP数据
df3 = pd.read_excel(r'D:\Limmo\Python数分\实验数据\宏观数据\GDP\2018各地区生产总值.xls')
# 清洗数据，提取地区和2018年GDP
df3 = df3.iloc[8:, [0, 2]]  # 第0列是地区，第2列是GDP
df3.columns = ['地区', '2018']
df3['地区'] = df3['地区'].str.replace(r'\s+', '', regex=True)# 删除文字中的空格
df3.reset_index(drop=True, inplace=True)# 删除原索引
df3 = df3.drop([31, 32, 33])
print("2018年GDP数据：")
print(df3.head())
print()

# 4. 合并所有年份数据得到2009-2018年GDP数据
# 先合并df1和df2
df_gdp = pd.merge(df1, df2, on='地区', how='outer')
# 再合并df3
df_gdp = pd.merge(df_gdp, df3, on='地区', how='outer')
print("2009-2018年GDP数据：")
print(df_gdp.head())

# 保存到excel
df_gdp.to_excel('ex2_4GDP.xlsx', index=False)
print("GDP数据已保存到 ex2_4GDP.xlsx")
print()

# （二）知识产权年鉴数据、人口数据、GDP数据、进出口总额、FDI清洗及整合 ==========

# 1. 读取知识产权年鉴数据
df1 = pd.read_excel('D:\Limmo\Python数分\实验数据\宏观数据\知识产权年鉴数据.xlsx')
print("知识产权年鉴数据原始列名：")
print(df1.columns.tolist())
print()

# 2. 选择列 重命名列名
df_selected = df1[['Unnamed: 0', 'Unnamed: 1', '结案率', '专利_申请量', '专利_授权量', '职务专利授权量百分比']].copy()
print("选取指定列后的数据：")
print(df_selected.head())
print()

# 3. 填充年份列
df_selected.rename(columns={'Unnamed: 0': '年份', 'Unnamed: 1': '地区'}, inplace=True)
df_selected['年份'] = df_selected['年份'].fillna(method='ffill')
df_selected['年份'] = df_selected['年份'].astype(int)
print("重命名、填充年份后的数据：")
print(df_selected.head())
print()

# 4. 处理结案率：将inf替换为0，大于1的替换为1
df_selected['结案率'] = df_selected['结案率'].replace([np.inf, -np.inf], 0)
df_selected.loc[df_selected['结案率'] > 1, '结案率'] = 1
print("处理结案率后的数据：")
print(df_selected.head())
print()

# 5. 读取各省人口数据
df_population = pd.read_excel('D:\Limmo\Python数分\实验数据\宏观数据\人口\各省人口数（2009-2018）.xlsx')
print("人口数据原始格式：")
print(df_population.head())
print()

# 6. 使用melt函数将人口数据从宽格式转换为长格式
df_population_long = pd.melt(df_population, id_vars=['地区'], var_name='年份', value_name='人口')
df_population_long['年份'] = df_population_long['年份'].astype(int)
df_population_long['人口'] = 10000*df_population_long['人口']
print("转换后的人口数据（长格式）：")
print(df_population_long.head())
print()

# 7. 整合知识产权数据和人口数据
df_combined = pd.merge(df_selected, df_population_long, on=['地区', '年份'], how='inner')
# 删除2008年数据
df_combined = df_combined[df_combined['年份'] != 2008].reset_index(drop=True)
print("整合后的数据前10行：")
print(df_combined.head(10))
print()

# 8. 整合GDP数据、进出口总额、FDI等

# 读取GDP数据
df_gdp = pd.read_excel('D:\Limmo\Python数分\实验二\ex2_4GDP.xlsx')
print("GDP数据：")
print(df_gdp.head())
print()
# 使用melt函数将GDP数据从宽格式转换为长格式
df_gdp_long = pd.melt(df_gdp, id_vars=['地区'], var_name='年份', value_name='GDP')
df_gdp_long['年份'] = df_gdp_long['年份'].astype(int)
df_gdp_long = df_gdp_long[df_gdp_long['年份'].isin([2009,2010,2011,2012,2013,2014,2015,2016,2017,2018])]
df_gdp_long['GDP'] = 100000000*df_gdp_long['GDP']
print("转换后的GDP数据（长格式）：")
print(df_gdp_long.head())
print()
# 合并到主数据集
df_combined = pd.merge(df_combined, df_gdp_long, on=['地区', '年份'], how='left')
print("已整合GDP数据")

# 读取各省进出口总额数据
df_trade = pd.read_excel('D:\Limmo\Python数分\实验数据\宏观数据\进出口2009~2025.xlsx',usecols=[0,1,2])
df_trade.columns.values[0] = '地区'
df_trade.columns.values[2] = '进出口总额'
df_trade.iloc[:,0] = df_trade.iloc[:,0].str.replace('省', '').str.replace('市', '').str.replace('自治区', '').str.replace('壮族', '').str.replace('回族', '').str.replace('维吾尔', '')
df_trade.iloc[:,1] = df_trade.iloc[:,1].str.replace('年', '')
# 删除2009-2018年以外的行 删除每17行中的前7行
for i in range(0, len(df_trade), 17):
    df_trade = df_trade.drop(index=[0+i, 1+i, 2+i, 3+i, 4+i, 5+i, 6+i])
df_trade.reset_index(drop=True, inplace=True)
df_trade['年份'] = df_trade['年份'].astype(int)
# 将千美元转换为元人民币（汇率6.5286）
df_trade['进出口总额'] = 6.5286*1000*df_trade['进出口总额']
# 合并到主数据集
df_combined = pd.merge(df_combined, df_trade, on=['地区', '年份'], how='left')
print("已整合进出口总额数据")

# 读取外商直接投资FDI数据
df_fdi = pd.read_excel('D:\Limmo\Python数分\实验数据\宏观数据\外商投资2009~2025.xlsx',usecols=[0,1,3])
df_fdi.columns.values[0] = '地区'
df_fdi.columns.values[2] = 'FDI'
df_fdi.iloc[:,0] = df_fdi.iloc[:,0].str.replace('省', '').str.replace('市', '').str.replace('自治区', '').str.replace('壮族', '').str.replace('回族', '').str.replace('维吾尔', '')
df_fdi.iloc[1:,1] = df_fdi.iloc[1:,1].str.replace('年', '')
# 删除2009-2018年以外的行 删除每17行中的前7行
for i in range(0, len(df_fdi), 17):
    df_fdi = df_fdi.drop(index=[0+i, 1+i, 2+i, 3+i, 4+i, 5+i, 6+i])
df_fdi.reset_index(drop=True, inplace=True)
df_fdi['年份'] = df_fdi['年份'].astype(int)
# 将百万美元转换为元人民币（汇率6.5286）
df_fdi['FDI'] = 6.5286*1000000*df_fdi['FDI']
# 合并到主数据集
df_combined = pd.merge(df_combined, df_fdi, on=['地区', '年份'], how='left')
print("已整合FDI数据")

# 保存最终整合的数据
df_combined.to_excel('ex2_4整合后的面板数据.xlsx', index=False)
print("\n最终整合数据已保存到 ex2_4整合后的面板数据.xlsx")
print("最终数据形状：", df_combined.shape)
print("最终数据：", df_combined.head())

<string>:21: SyntaxWarning: invalid escape sequence '\{'
<>:21: SyntaxWarning: invalid escape sequence '\{'
<>:21: SyntaxWarning: invalid escape sequence '\L'
<>:63: SyntaxWarning: invalid escape sequence '\L'
<>:90: SyntaxWarning: invalid escape sequence '\L'
<>:114: SyntaxWarning: invalid escape sequence '\L'
<>:131: SyntaxWarning: invalid escape sequence '\L'
<>:148: SyntaxWarning: invalid escape sequence '\L'
<string>:21: SyntaxWarning: invalid escape sequence '\{'
<>:21: SyntaxWarning: invalid escape sequence '\{'
<>:21: SyntaxWarning: invalid escape sequence '\L'
<>:63: SyntaxWarning: invalid escape sequence '\L'
<>:90: SyntaxWarning: invalid escape sequence '\L'
<>:114: SyntaxWarning: invalid escape sequence '\L'
<>:131: SyntaxWarning: invalid escape sequence '\L'
<>:148: SyntaxWarning: invalid escape sequence '\L'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_11184\2558966618.py:21: SyntaxWarning: invalid escape sequence '\{'
  file_path = f'D:\Limmo\Python数分\实验数据\宏观数据\GD

2009-2013年GDP数据：
    地区      2009      2010      2011      2012      2013
0   北京  12153.03  14113.58  16251.93  17879.40  19500.56
1   天津   7521.85   9224.46  11307.28  12893.88  14370.16
2   河北  17235.48  20394.26  24515.76  26575.01  28301.41
3   山西   7358.31   9200.86  11237.55  12112.83  12602.24
4  内蒙古   9740.25  11672.00  14359.88  15880.58  16832.38

2014-2017年GDP数据：
    地区     2014     2015     2016     2017
0   上海  23567.7  25123.5  28178.7  30633.0
1   云南  12814.6  13619.2  14788.4  16376.3
2  内蒙古  17769.5  17831.5  18128.1  16096.2
3   北京  21330.8  23014.6  25669.1  28014.9
4   吉林  14631.4  15507.9  14776.8  14944.5

2018年GDP数据：
    地区      2018
0   北京  30319.98
1   天津  18809.64
2   河北  36010.27
3   山西  16818.11
4  内蒙古  17289.22

2009-2018年GDP数据：
    地区      2009      2010      2011      2012      2013     2014     2015  \
0   上海  15046.45  17165.98  19195.69  20181.72  21602.12  23567.7  25123.5   
1   云南   6169.75   7224.18   8893.12  10309.47  11720.91  12814.6  13619.2  